# LangChain 실험 플레이그라운드 — 사용법

밈/신조어 키워드에 대해 질문 -> 벡터 검색(dense/sparse/융합) -> 프롬프트 조립 -> LLM 답변까지,
각 셀을 독립적으로 재실행하며 실험하기 위한 노트북입니다.

## 실행 전 준비

이 노트북은 기본적으로 **EC2 서버(진짜 운영 데이터)** 를 봅니다. 그러려면 별도 터미널에서 SSH 터널이 먼저 떠 있어야 합니다:
```
ssh -i ~/.ssh/id_ed25519 -L 27017:localhost:27017 -L 36333:localhost:6333 ec2-user@100.29.36.216
```
(mongo는 27017 그대로, qdrant는 EC2용으로 `36333`을 씀 — `6333`은 Windows 예약 포트라 로컬에서 못 열고, `16333`은 로컬 docker qdrant가 이미 쓰고 있어서 겹치지 않게 분리했습니다.)

## 실행 순서

1번(환경설정) → 2번(키워드 선택) → 2.5번(유행 상태, 선택) → 3번(질문+임베딩) → 4번(검색) → 5번(검색결과 확인) → 6번(프롬프트) → 7번(LLM 호출) 순서로 **한 번 끝까지** 실행하세요.

그다음부터는 **3번 ~ 7번만** 값을 바꿔가며 반복 재실행하면 됩니다 (1, 2번은 한 번만 실행하면 됨).

## 셀별로 바꿀 수 있는 값

| 셀 | 변수 | 용도 |
|---|---|---|
| 1. 환경설정 | `USE_EC2` | `True`=EC2 진짜 데이터(터널 필요), `False`=로컬 docker 테스트 데이터 |
| 2. 키워드 선택 | `KEYWORD` | 분석할 밈/신조어. 셀 실행하면 임베딩 완료된 키워드 목록이 출력됨 |
| 2.5. 유행 상태 | `INCLUDE_TREND` | 네이버 데이터랩 유행 상태를 프롬프트에 넣을지 여부 |
| 3. 질문+임베딩 | `QUESTION` | LLM에게 물어볼 자유 질문 |
| 4. 검색 파라미터 | `TOP_K` | 근거로 가져올 청크 개수 |
| 4. 검색 파라미터 | `SOURCES` | 검색 대상 소스 제한. 예: `["dcinside","natepann","namuwiki"]`로 tavily/youtube 제외. `None`=전체 |
| 6. 프롬프트 작성 | `PROMPT_TEMPLATE` | LLM에게 보낼 지시문 자체를 수정 (`{keyword}`/`{trend_info}`/`{context}`/`{question}` 자리표시자는 유지) |
| 7. LLM 호출 | `MODEL`/`TEMPERATURE`/`TOP_P` | 같은 프롬프트로 답변이 어떻게 달라지는지 비교 |

## 5번 셀(검색결과 확인) 읽는 법

- `[D]`/`[S]`/`[DS]`: FUSED 결과가 dense 검색에서 왔는지, sparse에서 왔는지, 둘 다인지
- `⚠️본문에 키워드 없음`: 그 청크에 `KEYWORD`가 실제로 안 들어있다는 뜻 — **크롤링 오염 의심** (아래 참고)

## 알려진 이슈

일부 키워드(특히 tavily/youtube 소스)는 크롤링 때 관련 없는 글이 섞여 들어가 있습니다 (예: "야르" 검색에 "골반통신" 밈 설명 글이 딸려온 사례). `SOURCES`로 특정 소스를 빼고 비교해보면 어느 소스가 원인인지 확인할 수 있지만, 소스를 통째로 빼면 그 소스의 멀쩡한 자료까지 같이 빠지는 트레이드오프가 있습니다 — 아직 문서 단위로 걸러내는 근본 해법은 적용 전입니다.

## 1. 환경 설정

**이 셀이 하는 일**: OS 환경변수, 인코딩, import를 준비합니다.

**바꿀 것**: `USE_EC2` — EC2 진짜 데이터를 보려면 `True`(기본값), 로컬 docker에 테스트 데이터를 직접 채웠을 때만 `False`.

**주의**:
- `.env`의 `MONGODB_URI`/`QDRANT_HOST`는 `mongo`/`qdrant`라는 docker-compose 내부 네트워크 호스트명을 가리키며 `mimori-flask` 컨테이너 안에서만 resolve됩니다. `.env` 파일 자체는 건드리지 마세요(도커 컨테이너들이 깨집니다) — 이 셀이 로컬 커널일 때 자동으로 `localhost`로 대체합니다.
- `USE_EC2=True`로 쓰려면 아래 SSH 터널이 먼저 떠 있어야 합니다:
  ```
  ssh -i ~/.ssh/id_ed25519 -L 27017:localhost:27017 -L 36333:localhost:6333 ec2-user@100.29.36.216
  ```
  (mongo는 27017 그대로, qdrant는 EC2용으로 36333을 씀. `6333`은 Windows가 예약한 포트 범위라 로컬에서 못 열고, `16333`은 로컬 docker qdrant가 이미 쓰고 있어서 겹치지 않게 `36333`을 씀.)

In [1]:
import sys
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# 이 커널이 docker 컨테이너 밖(로컬)에서 실행 중이면 .env의 mongo/qdrant 호스트명이
# DNS로 안 풀립니다. 컨테이너 안(env_file로 이미 mongo/qdrant가 주입된 상태)에서 실행
# 중이면 아래 setdefault는 아무 효과가 없고, 로컬 커널일 때만 값이 대체됩니다.
#
# USE_EC2 = True: EC2 서버(진짜 운영 데이터)에 SSH 터널로 붙습니다.
#   ssh -i ~/.ssh/id_ed25519 -L 27017:localhost:27017 -L 36333:localhost:6333 ec2-user@100.29.36.216
#   (mongo는 27017 그대로, qdrant는 EC2용으로 36333을 씀 — 로컬 docker qdrant가 16333을 이미
#    쓰고 있고, 6333 자체는 Windows가 예약한 포트 범위라 로컬에서 못 엽니다.)
# USE_EC2 = False: 로컬 docker-compose qdrant(16333)를 봅니다. 로컬에서 직접 크롤링/전처리/
#   임베딩을 돌려서 테스트 데이터를 채웠을 때만 씁니다.
USE_EC2 = True

os.environ.setdefault("MONGODB_URI", "mongodb://localhost:27017")
os.environ.setdefault("QDRANT_HOST", "localhost")
os.environ["QDRANT_PORT"] = "36333" if USE_EC2 else "16333"

# 이 노트북은 analysis/ 안에 있으므로 커널 작업 디렉토리는 analysis/ 입니다.
# 프로젝트 루트(analysis/의 상위 폴더)를 import 경로에 추가합니다.
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

if sys.platform == "win32":
    try:
        sys.stdin.reconfigure(encoding="utf-8")
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except AttributeError:
        pass

from analysis.pipeline import list_analyzable_keywords
from analysis.rag_pipeline import (
    search_relevant_chunks,
    search_dense_only,
    search_sparse_only,
)
from embedding.encoder import encode_batch
from config.config_cilent import NIM_KEY
from langchain_nvidia_ai_endpoints import ChatNVIDIA

print("설정 완료 (EC2 모드)" if USE_EC2 else "설정 완료 (로컬 모드)")

C:\workspace\memeory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


설정 완료 (EC2 모드)


## 2. 키워드 선택

**이 셀이 하는 일**: MongoDB에서 임베딩이 끝난 밈 키워드 목록을 가져와 보여줍니다.

**실험하려면**: 출력된 목록 중 하나를 골라 `KEYWORD` 변수에 문자열로 직접 대입한 뒤 재실행하세요.

In [2]:
keywords = list_analyzable_keywords()
print(f"분석 가능한 키워드 {len(keywords)}개:")
for kw in keywords:
    print(f"  - {kw}")

KEYWORD = "야르"
print(f"\n현재 선택된 KEYWORD = {KEYWORD!r}")

분석 가능한 키워드 13개:
  - 거제야호
  - 럭키비키
  - 롤린
  - 리센느
  - 뭔말알
  - 쌰갈
  - 아자스
  - 야르
  - 야호~
  - 에스파
  - 영크크
  - 이순신
  - 홍명보

현재 선택된 KEYWORD = '야르'


## 2.5. z-score / 유행 상태 확인

**이 셀이 하는 일**: 선택한 키워드(`KEYWORD`)의 최근 검색량 기반 유행 상태를 네이버 데이터랩에서 조회합니다.

**실험하려면**: 아래 코드 셀의 `INCLUDE_TREND`를 `True`/`False`로 바꾼 뒤, 이 셀을 먼저 재실행하고 "6. 프롬프트 작성" 셀을 실행해야 반영됩니다(`INCLUDE_TREND`는 이 셀에서 정의되므로, 값만 바꾸고 이 셀을 다시 실행하지 않으면 이전 값이 그대로 남아있습니다).

In [3]:
from trend.trend_service import format_trend_context

trend_info = format_trend_context(KEYWORD)
if trend_info:
    print(trend_info)
else:
    print("z-score/유행 상태를 가져오지 못했습니다 (NAVER API 키 미설정이거나 데이터 없음).")

INCLUDE_TREND = True

[trend_service] 카카오 지표 수집 실패, 제외 진행: KAKAO_REST_API_KEY 가 설정되지 않았습니다. .env 를 확인하세요.
[google_client] 요청 실패(ModuleNotFoundError("No module named 'pytrends'")), 2초 후 재시도 (1/2)
[google_client] 요청 실패(ModuleNotFoundError("No module named 'pytrends'")), 4초 후 재시도 (2/2)
[google_client] 최대 재시도 초과, google_z 제외: ModuleNotFoundError("No module named 'pytrends'")
[참고: 최근 검색/언급량 기반 유행 상태 앙상블 판정 — 정성적 분석의 보조 지표로만 활용]
상태: 감소 (robust z-score: -1.74, 반영 소스: naver)


## 3. 질문 입력 + 임베딩

**이 셀이 하는 일**: 자유 텍스트 질문을 BGE-M3로 dense 벡터 + sparse(lexical) 벡터로 변환합니다.

**실험하려면**: `QUESTION` 문자열만 바꾸면 됩니다. 질문이 바뀌면 검색되는 청크가 달라지므로, 이 셀부터 다시 실행해야 아래 결과에 반영됩니다. (임베딩 모델은 재실행마다 새로 로드하지 않도록 그대로 유지합니다 — LLM 호출은 NVIDIA API라 로컬 GPU와 충돌하지 않습니다.)

In [4]:
QUESTION = "이 밈은 왜 유행했나요?"

dense_vecs, lexical_weights = encode_batch([QUESTION])
dense_vec, sparse = dense_vecs[0], lexical_weights[0]

print(f"QUESTION = {QUESTION!r}")
print(f"dense_vec 길이 = {len(dense_vec)}")
print(f"sparse 항목 수 = {len(sparse)}")

[임베딩] BAAI/bge-m3 로드 중... (device=cuda)


Loading weights: 100%|██████████| 391/391 [00:01<00:00, 207.00it/s]


QUESTION = '이 밈은 왜 유행했나요?'
dense_vec 길이 = 1024
sparse 항목 수 = 10


## 4. 검색 파라미터 + 실행

**이 셀이 하는 일**: 같은 질문 벡터로 dense만 / sparse만 / RRF 융합, 세 가지 방식으로 Qdrant에서 관련 청크를 검색합니다.

**실험하려면**:
- `TOP_K`를 늘리거나 줄여서 더 많은/적은 근거 청크를 가져와볼 수 있습니다.
- `SOURCES`로 검색 대상 소스를 제한할 수 있습니다. 예를 들어 tavily/youtube가 오염됐다고 의심되면 `SOURCES = ["dcinside", "natepann", "namuwiki"]`로 빼고 돌려서, 답변이 좋아지는지 비교해보세요. `None`이면 전체 소스(`tavily`/`youtube`/`namuwiki`/`natepann`/`dcinside`)를 다 봅니다.

In [5]:
TOP_K = 5
SOURCES = None  # 예: ["dcinside", "natepann", "namuwiki"] 처럼 특정 소스만 검색할 수 있음 (None=전체)

dense_points = search_dense_only(KEYWORD, dense_vec, TOP_K, sources=SOURCES)
sparse_points = search_sparse_only(KEYWORD, sparse, TOP_K, sources=SOURCES)
fused_points = search_relevant_chunks(KEYWORD, dense_vec, sparse, TOP_K, sources=SOURCES)

print(f"SOURCES = {SOURCES!r}")
print(f"dense={len(dense_points)}, sparse={len(sparse_points)}, fused={len(fused_points)}")

SOURCES = None
dense=5, sparse=5, fused=5


## 5. 검색 결과 확인 ("왜 이게 뽑혔나" 진단 포함)

**이 셀이 하는 일**: dense / sparse / 융합(RRF) 검색 결과를 나란히 비교 출력합니다. 각 결과 옆에:
- `[D]`/`[S]`/`[DS]`: FUSED 결과가 dense에서만 왔는지, sparse에서만 왔는지, 둘 다에서 왔는지(RRF가 왜 이 청크를 올렸는지)
- `⚠️본문에 키워드 없음`: 이 청크의 본문/제목에 `KEYWORD` 문자열이 실제로 안 들어있으면 표시 — 크롤링 오염(다른 주제의 글이 이 키워드로 잘못 저장된 경우) 즉시 확인 가능

**실험하려면**: 이 셀 자체는 결과를 출력만 하므로 수정할 것이 없습니다 — 위 셀들(질문, TOP_K, 키워드)을 바꾸고 재실행해서 차이를 비교하세요. `⚠️` 표시가 많이 뜨면, 그 키워드는 크롤링 데이터 자체가 오염됐을 가능성이 높습니다 (실제로 `야르`에서 발견된 사례 — tavily가 "밈 뜻 유래" 같은 범용 검색어에 이끌려 전혀 다른 밈을 설명하는 글까지 가져와서 `야르`로 잘못 저장한 경우가 있었습니다).

In [6]:
def _print_points(label, points, dense_ids=None, sparse_ids=None):
    print(f"--- {label} ({len(points)}개) ---")
    for p in points:
        title = p.payload.get("title") or "제목 없음"
        url = p.payload.get("url") or "출처 없음"
        text = p.payload.get("text", "")

        origin = ""
        if dense_ids is not None and sparse_ids is not None:
            in_dense = p.id in dense_ids
            in_sparse = p.id in sparse_ids
            origin = "[" + ("D" if in_dense else "") + ("S" if in_sparse else "") + "] "

        contains_kw = KEYWORD in text or KEYWORD in title
        warn = "" if contains_kw else "  ⚠️본문에 키워드 없음(크롤링 오염 의심)"

        print(f"  {origin}score={p.score:.4f} | {title} ({url}){warn}")
        print(f"  {text[:150]}")
    print()

dense_ids = {p.id for p in dense_points}
sparse_ids = {p.id for p in sparse_points}

_print_points("DENSE", dense_points)
_print_points("SPARSE", sparse_points)
_print_points("FUSED(RRF)", fused_points, dense_ids, sparse_ids)

--- DENSE (5개) ---
  score=0.6518 | 요즘 유행하는 <골반통신> 밈 알아보기! (뜻, 유래, 챌린지) - 트로스트 커뮤니티 (https://trost.co.kr/community/jayuu/119004299)  ⚠️본문에 키워드 없음(크롤링 오염 의심)
  기업 전용 멘탈케어 프로그램을 도입하고 싶다면?

지금 넛지EAP 이용해보기

마음을 챙기는 습관,
트로스트 앱과 함께
만들어 보세요

trost app download qr code

궁금한 내용을 검색해보세요

## 자유게시판

# 요즘 유행하는 밈 알아보기! (뜻
  score=0.6319 | 밈 뜻 유래 총정리 (https://well-inform.tistory.com/83)  ⚠️본문에 키워드 없음(크롤링 오염 의심)
  밈은 시간과 공간을 초월하여 다양한 문화권에서 공통적으로 나타나는 현상입니다. 이는 밈이 특정 문화나 사회적 배경에 국한되지 않고, 보편적인 인간의 심리와 소통 방식을 반영하기 때문입니다. 밈은 종종 유머, 풍자, 사회적 메시지를 담고 있어 대중의 공감을 이끌어냅니다.
  score=0.6278 | 샤갈 뜻 유래 야르 밤티 완벽 정리 (https://sherry7777.tistory.com/entry/%EC%83%A4%EA%B0%88-%EB%9C%BB-%EC%9C%A0%EB%9E%98-%EC%95%BC%EB%A5%B4-%EB%B0%A4%ED%8B%B0-%EC%99%84%EB%B2%BD-%EC%A0%95%EB%A6%AC)
  이 밈이 이토록 강력하게 퍼진 이유는 특유의 '리듬감' 덕분입니다.
단순히 "기분 나빠"라고 말하는 것보다 "진짜 샤갈하네"라고 말할 때 느껴지는 말맛이 MZ세대의 취향을 저격한 것이죠.
이제는 대학생과 직장인들 사이에서도 황당한 상황을 풍자할 때 즐겨 쓰는 단어가 되
  score=0.6262 | 밈 뜻 유래 총정리 (https://well-inform.tistory.com/83)  ⚠️본문에 키워드 없음(크롤링 오염 의심)
  we

## 6. 프롬프트 작성

**이 셀이 하는 일**: 검색된 청크(`fused_points`)와 유행 상태 정보(`trend_info`, `INCLUDE_TREND`가 `True`일 때만)를 근거 자료로 넣어 LLM에게 실제로 전달할 프롬프트를 완성합니다.

**실험하려면**: `PROMPT_TEMPLATE` 문자열을 자유롭게 수정하세요(지시문 추가, 어조 변경, 출력 형식 지정 등). `{keyword}` / `{trend_info}` / `{context}` / `{question}` 네 자리표시자는 반드시 그대로 남겨둬야 합니다. `.format()`을 쓰므로, 프롬프트에 `{`/`}` 자체를 문자로 넣고 싶으면(예: JSON 출력 형식을 지시하는 경우) `{{`/`}}`로 이스케이프해야 합니다.

In [7]:
PROMPT_TEMPLATE = """당신은 밈/신조어 분석 전문가입니다.

키워드: {keyword}

{trend_info}
자료:
{context}

질문: {question}

위 자료를 근거로 답변하세요. 자료에 없는 내용은 추측하지 말고 모른다고 답하세요.
"""

context = "\n\n---\n\n".join(
    f"[출처: {p.payload.get('title') or '제목 없음'} / {p.payload.get('url') or '출처 없음'}]\n{p.payload.get('text', '')}"
    for p in fused_points
)
prompt = PROMPT_TEMPLATE.format(
    keyword=KEYWORD,
    trend_info=(trend_info if INCLUDE_TREND else ""),
    context=context,
    question=QUESTION,
)
print(prompt)

당신은 밈/신조어 분석 전문가입니다.

키워드: 야르

[참고: 최근 검색/언급량 기반 유행 상태 앙상블 판정 — 정성적 분석의 보조 지표로만 활용]
상태: 감소 (robust z-score: -1.74, 반영 소스: naver)
자료:
[출처: 요즘 유행하는 <골반통신> 밈 알아보기! (뜻, 유래, 챌린지) - 트로스트 커뮤니티 / https://trost.co.kr/community/jayuu/119004299]
기업 전용 멘탈케어 프로그램을 도입하고 싶다면?

지금 넛지EAP 이용해보기

마음을 챙기는 습관,
트로스트 앱과 함께
만들어 보세요

trost app download qr code

궁금한 내용을 검색해보세요

## 자유게시판

# 요즘 유행하는 밈 알아보기! (뜻, 유래, 챌린지)

2025.10.23 11:33

조회 3.1천추천 4스크랩 1

혹시 이라는 말 들어보셨나요?
요즘 한창 유행하는 밈이자 챌린지인데요

한번 정리해보았습니다!

## 밈 유래와 뜻 정리

이 밈은 크리에이터 퐁귀가 올린

영상에서 시작되었는데요

골반이 멈추지 않아 영상 보러가기

내 골반이 멈추지 않는 탓일까? ㅜ.ㅜ

하면서 골반을 계~속 흔드는 영상과

묘하게 촌스럽고 웃긴 편집

그리고 말도 안되는 내용 때문에

인기를 얻게 되었어요 (ㅋㅋㅋ)

정말 신기한 건,

이 밈에서 쓰인 배경 음악이

AOA의 짧은 치마 (inst) 인데요

이 노래까지 역주행을 하고 있다고 해요..

ㅋㅋㅋ

---

[출처: 밈 뜻 유래 총정리 / https://well-inform.tistory.com/83]
wellinform 2025. 1. 10.

밈 뜻과 유래 알아보기

밈(Meme)은 현대 디지털 사회에서 독특한 문화적 현상으로 자리 잡았습니다. 인터넷과 소셜 미디어의 발달로 빠르게 확산되는 이 단어는 단순한 유행을 넘어 인간의 생각, 감정, 그리고 사회적 메시지를 효과적으로 전달하는 도구로 활용되고 있습니다. 밈은 이미지, 텍스트, 비디오 등 다양한 형태로 나타나

## 7. LLM 호출

**이 셀이 하는 일**: 완성된 프롬프트를 NVIDIA API(ChatNVIDIA)로 보내고 답변을 받아 출력합니다.

**실험하려면**: `MODEL` / `TEMPERATURE` / `TOP_P` 값을 바꿔서, 같은 프롬프트에도 답변이 어떻게 달라지는지 비교해보세요.

In [8]:
assert NIM_KEY, "NIM_KEY가 .env에 설정되어 있지 않습니다"

MODEL = "deepseek-ai/deepseek-v4-flash"
TEMPERATURE = 1
TOP_P = 0.95

llm_client = ChatNVIDIA(
    model=MODEL,
    api_key=NIM_KEY,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_completion_tokens=16384,
    timeout=6000,
)

response = llm_client.invoke([{"role": "user", "content": prompt}])
print(response.content)

주어진 자료에는 '야르'라는 키워드에 대한 내용이 전혀 포함되어 있지 않습니다. 따라서 해당 밈이 왜 유행했는지에 대해 자료를 근거로 답변할 수 없습니다.
